# 🌐 IA Generativa con ArangoDB
Este notebook muestra cómo usar ArangoDB con documentos, grafos y embeddings para búsqueda semántica.

In [ ]:
# Instalar driver si no está presente
# !pip install python-arango numpy
from arango import ArangoClient
import numpy as np


## 🔌 Conexión a ArangoDB

In [ ]:
# Crear cliente y conexión
client = ArangoClient()
db = client.db('_system', username='root', password='')

# Crear base de datos si no existe
if not db.has_database('ia_generativa'):
    db.create_database('ia_generativa')

# Conectarse a la base creada
db = client.db('ia_generativa', username='root', password='')


## 🧱 Crear colecciones de documentos y grafos

In [ ]:
# Crear colecciones
if not db.has_collection('prompts'):
    db.create_collection('prompts')

if not db.has_collection('usuarios'):
    db.create_collection('usuarios')

# Crear grafo
if not db.has_graph('interacciones'):
    graph = db.create_graph('interacciones')
    graph.create_vertex_collection('usuarios')
    graph.create_vertex_collection('prompts')
    graph.create_edge_definition(
        edge_collection='genero',
        from_vertex_collections=['usuarios'],
        to_vertex_collections=['prompts']
    )
else:
    graph = db.graph('interacciones')


## ➕ Insertar documentos con embeddings simulados

In [ ]:
from random import random

usuarios = db.collection('usuarios')
prompts = db.collection('prompts')
genero = graph.edge_collection('genero')

# Usuario ejemplo
u = usuarios.insert({'_key': 'u1', 'nombre': 'Ana'}, overwrite=True)

# Generar 3 prompts con embeddings aleatorios (simulación)
for i in range(3):
    emb = [round(random(), 4) for _ in range(5)]
    p = prompts.insert({
        '_key': f'p{i+1}',
        'texto': f"Prompt ejemplo {i+1}",
        'embedding': emb
    }, overwrite=True)
    genero.insert({'_from': 'usuarios/u1', '_to': f'prompts/p{i+1}'}, overwrite=True)


## 🔍 Búsqueda semántica por similitud de coseno

In [ ]:
# Función para similitud de coseno
def cosine_similarity(vec1, vec2):
    a = np.array(vec1)
    b = np.array(vec2)
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

# Supongamos que este es el nuevo prompt (vector de entrada)
query_vector = [0.5, 0.5, 0.5, 0.5, 0.5]

# Buscar los documentos más similares
all_prompts = prompts.all()
resultados = []
for doc in all_prompts:
    score = cosine_similarity(query_vector, doc['embedding'])
    resultados.append((doc['texto'], round(score, 3)))

# Mostrar los más similares
resultados = sorted(resultados, key=lambda x: -x[1])
for texto, sim in resultados:
    print(f"Sim: {sim} - {texto}")


## ✅ Conclusión
Este notebook demuestra cómo usar ArangoDB con grafos y embeddings para apoyar sistemas generativos como motores de recomendación o generación contextual.